In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import requests
from urllib.parse import quote

In [2]:
HIST_START = pd.Timestamp("2025-10-13")
WEEK_END = pd.Timestamp("2026-10-18")
DAYS = pd.date_range(HIST_START, WEEK_END, freq="D")
UA = {"User-Agent": "DS3-IndicatorLab/1.0 (jms.mulyono@gmail.com)"}

In [10]:
TITLES = {
    "AAPL": "Apple_Inc.",
    "MSFT": "Microsoft",
    "NVDA": "Nvidia",
    "TSLA": "Tesla,_Inc.",
    "AMZN": "Amazon_(company)",
    "META": "Meta_Platforms",
    "GOOGL": "Alphabet_Inc.",
    "AMD": "AMD",
    "NFLX": "Netflix",
    "JPM": "JPMorgan_Chase",
}



In [11]:
r = requests.get(
    "https://en.wikipedia.org/w/api.php",
    headers=UA,
    params={
        "action": "query",
        "titles": "|".join(TITLES.values()),
        "redirects": 1,
        "format": "json",
        "formatversion": 2,
    },
)

In [12]:

q = r.json()["query"]
print("redirects:", q.get("redirects"))
print("missing:", [p["title"] for p in q["pages"] if p.get("missing")])
print([p["title"] for p in q["pages"]])

redirects: None
missing: []
['Apple Inc.', 'AMD', 'Microsoft', 'Nvidia', 'Amazon (company)', 'Netflix', 'JPMorgan Chase', 'Tesla, Inc.', 'Alphabet Inc.', 'Meta Platforms']


In [14]:
%load_ext autoreload
%autoreload 2
from src.wikipedia import pageviews

v = pageviews("Apple_Inc.", HIST_START, WEEK_END)
print(len(v), v["2026-09-09"])   # expect 371, 22115

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
371 22115


In [15]:
from pathlib import Path
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def load_views(ticker, title):
    f = RAW_DIR / f"{ticker}_views.csv"
    if f.exists():
        return pd.read_csv(f, index_col=0, parse_dates=True).iloc[:, 0]
    s = pageviews(title, HIST_START, WEEK_END).rename(ticker)
    s.to_csv(f)
    return s

daily = pd.concat([load_views(t, ti) for t, ti in TITLES.items()], axis=1)
print(daily.shape, daily.loc["2026-10-16", "TSLA"])   # expect (371, 10), 12751

(371, 10) 12751
